# Advanced Resume RAG experiments

Use reusable modules and measured artifacts. The Djinni subset uses a fixed-seed random sample of original row positions. There are no trusted resume?job judgments; ANN agreement is not semantic relevance. Research profiles stay local.


## Foundations

Tokenization maps text to tokens; a Transformer encoder builds contextual token representations; pooling combines them into a vector. Normalized dot products equal cosine similarity. The encoder windows long inputs. BM25 combines term frequency, inverse document frequency, and length normalization. HNSW searches a proximity graph. RRF sums 1/(k+rank) over ranked lists, avoiding direct comparison of scores.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from src.retrieval import bm25, rrf
from src.evaluation import judged_metrics
results = root / "results"


## Dataset and evaluation constraints

The upstream Djinni release has 210,250 English candidate rows and 141,897 English job rows. The local experiment uses the 10,000 profiles and 200 jobs sampled without replacement using seed 1729. The earlier source-ordered sample is retained for comparison, but 518 IDs overlap. Its labels are not human relevance judgments. See DATASET_AUDIT.md for provenance, privacy, rights, and the smaller benchmark.


In [ ]:
ann = json.loads((results / "djinni_random_1729_ann.json").read_text(encoding="utf-8-sig"))
pd.Series({key: ann[key] for key in ["query_count", "resume_chunks", "qdrant_indexed_vectors", "ann_recall_at_10", "ann_recall_at_20", "faiss_ms_p50", "faiss_ms_p95", "qdrant_ms_p50", "qdrant_ms_p95"]})


In [ ]:
device = json.loads((results / "device_comparison.json").read_text(encoding="utf-8-sig"))
latency = pd.DataFrame({mode:{target:device[mode][target]["median_ms"] for target in ["cpu","cuda"]} for mode in ["embedding","reranking"]})
latency.plot.bar(rot=0, ylabel="Median milliseconds per batch of 8", title="Local inference: three measured runs")
plt.tight_layout()
plt.show()


## HNSW and timing

An isolated filtered query on the random collection incremented the HNSW graph-search counter in seven segments. Qdrant can choose different plans for other queries. The recorded readiness timer expired with red status after a Windows rename error; green status was restored only after restarting the server, so that timer is not an index-build duration. See PROGRESS.md for the server version, construction timing, sampling and resource limitations.


In [ ]:
timing_path = results / "djinni_random_1729_index.json"
print(json.loads(timing_path.read_text(encoding="utf-8-sig")) if timing_path.exists() else "Index construction measurement pending")


## Generation and agent smoke checks

The public-domain U.S. Bureau of Labor Statistics occupation passage is a demonstration fixture, not a resume, vacancy dataset, or evaluation set. Naive, hybrid, advanced and stepwise outputs are checked for source-reference validity. One agent run records tool selection and citation checks. Claim entailment and population-level answer quality require independent human review.


In [ ]:
files = ["bls_naive.json", "bls_hybrid.json", "bls_advanced.json", "bls_stepwise.json", "bls_agent_validated.json"]
summary = []
for name in files:
    path = results / name
    if path.exists() and path.stat().st_size:
        value = json.loads(path.read_text(encoding="utf-8-sig"))
        summary.append({"configuration":name, "source_checks_passed":value.get("source_checks_passed"), "stopped":value.get("stopped")})
pd.DataFrame(summary)


## Pending

Few-shot tests require independently reviewed authentic examples; no examples have been generated to fill the gap. Relevance ablations, RAG and prompt quality comparisons require independently checked judgments and appropriate rights. Do not label unjudged pairs as irrelevant or use an LLM to fill the human annotation worksheet.


## Duplicate audit and preserved source-order comparison

Near-duplicate screening uses a conservative SimHash filter and may miss matches; 518 source IDs overlap the earlier sample, so the two runs are not independent validation splits.


In [ ]:
audit=json.loads((results/"sample_duplicate_audit.json").read_text(encoding="utf-8-sig"))
old=json.loads((results/"djinni_scale_ann.json").read_text(encoding="utf-8-sig"))
new=json.loads((results/"djinni_random_1729_ann.json").read_text(encoding="utf-8-sig"))
print("Duplicate audit:",audit)
pd.DataFrame({"source_order":old,"seeded_sample":new}).loc[["query_count","resume_chunks","ann_recall_at_10","ann_recall_at_20","qdrant_ms_p50","qdrant_ms_p95"]]


## Retrieval ablation demonstration

All five paths were run against the same single public BLS question, and only that question was sent externally for expansion. This tests mechanics, not semantic quality; controlled relevance comparisons require independent human judgments. The worksheet is local and its grading fields are blank.


In [ ]:
ablation=json.loads((results/"ablation_demo.json").read_text(encoding="utf-8-sig"))
print(ablation)
